# Control-defined coarse developmental-state atlas

This notebook builds a publication-quality coarse developmental-state atlas for the D0–D3 CC-Perturb-seq experiment **without using target→program CRT effects, target→program differential effects, or perturbation-driven clusters**.

State-definition priority:

1. approved states from an independent D0–D3 reference;
2. label transfer from an independent D0–D3 reference;
3. clusters learned from pure control cells only.

Pure controls are defined as cells carrying at least one safe-targeting or non-targeting guide and **zero** real-targeting guides. Cells carrying both control and real-targeting guides are excluded from the training set but retained in the final all-cell projection.

The notebook is intentionally two-stage:

- **Review stage:** audit the objects, define controls, fit the fixed atlas source, cluster source cells when needed, calculate marker/timing evidence, and export an annotation review table.
- **Finalize stage:** apply an author-reviewed cluster→state map, project all cells into the fixed atlas, and export the final panels and source tables.


## 0. Sherlock paths and package setup

The final processed CC-Perturb-seq query path is prefilled in both YAML files. The independent reference is left `null`, so the atlas is learned from strict pure-control cells. This workflow is D0–D3 only.


In [ ]:
from pathlib import Path
import os, sys, yaml, pandas as pd

PACKAGE_DIR = Path.cwd()
if not (PACKAGE_DIR / "src").exists():
    PACKAGE_DIR = Path.cwd().parent
sys.path.insert(0, str(PACKAGE_DIR / "src"))

CONFIG_TEMPLATE = PACKAGE_DIR / "config" / "control_defined_state_atlas_config.yaml"
CONFIG_LOCAL = PACKAGE_DIR / "config" / "control_defined_state_atlas_config.local.yaml"

if not CONFIG_LOCAL.exists():
    CONFIG_LOCAL.write_text(CONFIG_TEMPLATE.read_text())
    print(f"Created prefilled local config: {CONFIG_LOCAL}")
else:
    print(f"Using prefilled local config: {CONFIG_LOCAL}")

cfg = yaml.safe_load(CONFIG_LOCAL.read_text())
cfg


## 1. Optional discovery of candidate H5AD/reference objects

Run the read-only discovery script if the final paths are not known. It inspects H5AD metadata in backed mode and does not modify any files.

In [ ]:
# Discovery is optional because the query path is already known.
DISCOVERY_ROOTS = [
    "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb",
    "/oak/stanford/groups/engreitz/Users/tri",
    "/scratch/users/edtng",
]
DISCOVERY_OUT = PACKAGE_DIR / "discovery_audit"

# Uncomment only if the configured file was moved:
# cmd = [sys.executable, str(PACKAGE_DIR / "scripts" / "00_discover_single_cell_inputs.py"),
#        "--roots", *DISCOVERY_ROOTS, "--outdir", str(DISCOVERY_OUT),
#        "--max-depth", "10", "--inspect-h5ad"]
# import subprocess; subprocess.run(cmd, check=True)


## 2. Verify the live query object

The configured query is the integrated D0–D3 CC-Perturb-seq H5AD used in the uploaded project notebooks. This cell checks that the OAK file is visible from the current Sherlock session.


In [ ]:
from control_defined_state_atlas import load_config

cfg, _ = load_config(CONFIG_LOCAL)
for key in ["query_h5ad", "reference_h5ad", "guide_metadata_tsv", "output_dir"]:
    print(f"{key}: {cfg.get(key)}")

query_path = Path(cfg["query_h5ad"])
if not query_path.exists():
    raise FileNotFoundError(
        f"Configured query H5AD is not visible: {query_path}\n"
        "Check OAK access with ls -lh. If the object moved, update both YAML files in config/."
    )
print(f"Query object found: {query_path}")
if cfg.get("reference_h5ad"):
    reference_path = Path(cfg["reference_h5ad"])
    assert reference_path.exists(), f"Reference H5AD does not exist: {reference_path}"


## 3. Review stage

This stage fits the atlas **only** on the independent reference or pure controls and writes:

- `data_audit.tsv`
- `control_cell_definition_qc.tsv`
- `control_counts_by_day_and_replicate.tsv`
- `control_state_annotation_review.tsv`
- `control_cluster_marker_scores.tsv`
- `atlas_feature_genes.tsv`
- a provisional Candidate A review plot

The run stops rather than silently broadening controls when pure controls are too sparse.

In [ ]:
from control_defined_state_atlas import run_review

OUTDIR = run_review(CONFIG_LOCAL)
print(f"Review outputs written to: {OUTDIR}")


In [ ]:
audit = pd.read_csv(Path(OUTDIR) / "data_audit.tsv", sep="\t")
control_counts = pd.read_csv(Path(OUTDIR) / "control_counts_by_day_and_replicate.tsv", sep="\t")
review = pd.read_csv(Path(OUTDIR) / "control_state_annotation_review.tsv", sep="\t")

print("DATA AUDIT")
display(audit)
print("PURE CONTROL COUNTS")
display(control_counts)
print("CLUSTER / STATE REVIEW")
display(review)


## 4. Author review of coarse state labels

Review each source cluster using:

- dominant observed day;
- multiple-marker evidence;
- top marker genes;
- replicate composition;
- the provisional UMAP.

Do **not** approve a state identity from one marker alone. Copy `templates/approved_cluster_state_map.tsv` to the output directory, add one row per source cluster, and set `approved_cluster_state_map_tsv` in the local YAML config.

Recommended conservative labels:

- hiPSC
- Primitive streak
- Posterior / alternative primitive streak
- Lateral mesoderm
- Vascular / pre-EC progenitor
- Endothelial cell
- Contractile / mesenchymal alternative
- Other / unresolved


In [ ]:
# Generate a convenient review template from the current clusters.
review_template = review[["source_cluster", "provisional_state", "annotation_confidence", "dominant_day", "n_cells"]].copy()
review_template = review_template.rename(columns={"provisional_state": "approved_state"})
review_template["author_reviewed"] = False
review_template["notes"] = ""
review_path = Path(OUTDIR) / "approved_cluster_state_map.tsv"
review_template.to_csv(review_path, sep="\t", index=False)
print(f"Edit this file before finalize mode: {review_path}")
display(review_template)


After editing the mapping, update the YAML configuration:

```yaml
approved_cluster_state_map_tsv: /absolute/path/to/control_defined_state_atlas_results/approved_cluster_state_map.tsv
```

Optionally also provide approved state/day palettes. The fallback palettes are clearly marked provisional.

## 5. Finalize stage

This stage trains the fixed source-state classifier, assigns every query cell, and exports Candidate A, Candidate B, the selected main-text panel, source tables, legend, methods, and QC files. All perturbed cells are transformed into the already fitted atlas and cannot influence state boundaries.

In [ ]:
from control_defined_state_atlas import run_finalize

# Re-read CONFIG_LOCAL after editing the approved mapping path.
OUTDIR = run_finalize(CONFIG_LOCAL)
print(f"Final outputs written to: {OUTDIR}")


In [ ]:
assignments = pd.read_csv(Path(OUTDIR) / "control_state_assignments.tsv", sep="\t")
state_counts = pd.read_csv(Path(OUTDIR) / "state_counts_by_day_and_replicate.tsv", sep="\t")
marker_summary = pd.read_csv(Path(OUTDIR) / "state_marker_summary.tsv", sep="\t")
qc = pd.read_csv(Path(OUTDIR) / "state_atlas_qc.tsv", sep="\t")

print("QC")
display(qc)
print("STATE COUNTS")
display(state_counts.head(30))
print("TOP MARKERS")
display(marker_summary.groupby("state", group_keys=False).head(10))


## 6. Final visual inspection checklist

Open the exported PNG/PDF at intended main-text size and verify:

- labels do not overlap dense point regions;
- the state palette is consistent with approved project colors;
- rare or unresolved states are not overnamed;
- the day-composition strip remains legible;
- control/source and all-cell panels use the same fixed embedding;
- point layers are rasterized, while text and axes remain vector;
- the caption says **cross-sectional atlas**, not lineage tracing;
- silhouette is reported only as QC, not biological validation.


In [ ]:
expected = [
    "Fig2_control_defined_state_atlas.svg",
    "Fig2_control_defined_state_atlas.pdf",
    "Fig2_control_defined_state_atlas.png",
    "control_state_assignments.tsv",
    "control_state_annotation_review.tsv",
    "state_marker_summary.tsv",
    "state_counts_by_day_and_replicate.tsv",
    "state_color_map.tsv",
    "draft_panel_legend.md",
    "methods_note.md",
]
for name in expected:
    p = Path(OUTDIR) / name
    print(f"{'OK' if p.exists() else 'MISSING'}  {p}")
